# 🧠 Advanced RAG Agent

## Learning Objectives
In this notebook, you will learn:
1. **Query Enhancement** - Reformulate questions using conversation history for better retrieval
2. **Topic Validation** - Classify questions as on-topic or off-topic using structured LLM output
3. **Document Relevance Assessment** - Evaluate retrieved documents before generating answers
4. **Query Optimization Loop** - Retry retrieval with improved queries when initial results are poor
5. **Conversation Memory** - Maintain state across multi-turn conversations with LangGraph checkpointing

## Prerequisites
- Completion of Chapters 01-02 and `01_Simple_RAG_Agent`
- API keys configured in `.env` (Databricks)

> **Note**: This notebook builds a TechFlow Solutions support agent with 8 graph nodes, conditional routing, and an optimization retry loop - significantly more sophisticated than the simple RAG agent.

| Property | Value |
|---|---|
| Origin | LangGraph reference agentic-RAG pattern, formalised by Jeong et al. (Adaptive-RAG, 2024) |

---

## 🔧 Part 1: Environment Setup

We import all required libraries and initialize our helper functions. This notebook uses the project's `helpers` factory for LLM and embedding initialization, plus LangGraph's `MemorySaver` for conversation persistence.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Import Libraries and Helper Functions
# ============================================================================

import os, sys
import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from typing import TypedDict, List, Literal
from pydantic import BaseModel, Field
from IPython.display import Image, display

# --- LangChain / LangGraph ---
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables.graph import MermaidDrawMethod

# --- Project helpers ---
sys.path.append(os.path.abspath(".."))
from helpers.utils import get_llm, get_databricks_embeddings

load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---

## 📄 Part 2: Knowledge Base & Vector Store

We create an in-memory knowledge base for a fictional company, **TechFlow Solutions**, with documents covering pricing, services, company history, and support procedures. These are embedded and stored in a Chroma vector database for retrieval.

In [2]:
# ============================================================================
# KNOWLEDGE BASE: Create Documents and Vector Store
# ============================================================================

# Initialize the embedding model
embedding_model = get_databricks_embeddings()

# Create comprehensive technology support knowledge base
knowledge_documents = [
    Document(
        page_content="TechFlow Solutions offers three main service tiers: Basic Support ($29/month) includes email support and basic troubleshooting, Professional Support ($79/month) includes priority phone support and advanced diagnostics, Enterprise Support ($199/month) includes 24/7 dedicated support and custom integrations.",
        metadata={"source": "pricing_guide.pdf", "category": "pricing"},
    ),
    Document(
        page_content="Our cloud infrastructure services include: Virtual Private Servers starting at $15/month, Managed Databases from $45/month, Content Delivery Network at $0.08/GB, and Load Balancing services at $25/month. All services include 99.9% uptime guarantee.",
        metadata={"source": "infrastructure_pricing.pdf", "category": "services"},
    ),
    Document(
        page_content="TechFlow Solutions was founded in 2018 by Maria Rodriguez, a former Google engineer with 15 years of experience in cloud architecture. The company has grown from 3 employees to over 150 team members across 12 countries, specializing in enterprise cloud solutions.",
        metadata={"source": "company_history.pdf", "category": "company"},
    ),
    Document(
        page_content="Our technical support team operates 24/7 for Enterprise customers, business hours (9 AM - 6 PM EST) for Professional customers, and email-only support for Basic customers. Average response times: Enterprise (15 minutes), Professional (2 hours), Basic (24 hours).",
        metadata={"source": "support_procedures.pdf", "category": "support"},
    )
]

# Build vector database
vector_store = Chroma.from_documents(knowledge_documents, embedding_model)
document_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print(f"✅ Knowledge base created with {len(knowledge_documents)} documents")

✅ Knowledge base created with 4 documents


---

## 📊 Part 3: Define Conversation State

The state schema tracks the full conversation lifecycle: the current query, enhanced query, retrieved documents, topic classification, and optimization attempts. This is richer than the simple `AgentState` from notebook 01.

In [3]:
# ============================================================================
# CONVERSATION STATE: Define the Graph State Schema
# ============================================================================

class ConversationState(TypedDict):
    conversation_history: List[BaseMessage]  # Full conversation thread
    retrieved_documents: List[Document]      # Current retrieved documents
    topic_relevance: str                     # On-topic or off-topic classification
    enhanced_query: str                      # Reformulated question
    should_generate: bool                    # Whether to proceed with answer generation
    optimization_attempts: int               # Number of query refinement attempts
    current_query: HumanMessage              # User's current question

---

## 🧩 Part 4: Core Agent Components

This section defines the 6 node functions that power the advanced RAG pipeline:

1. **Query Enhancer** - Reformulates questions using conversation history
2. **Topic Validator** - Classifies on-topic vs. off-topic with structured output
3. **Content Retriever** - Fetches documents from the vector store
4. **Relevance Assessor** - Grades each document for relevance
5. **Response Generator** - Produces context-aware answers
6. **Query Optimizer** - Rewrites failed queries for retry

### Key Concepts:
- **Structured Output**: Pydantic models enforce typed LLM responses for classification
- **Query Enhancement**: Conversation history is folded into standalone search queries
- **Optimization Loop**: Failed retrievals trigger query rewriting (max 2 attempts)

### 4.1 🔍 Query Enhancer - Intelligent Question Reformulation

For follow-up questions, the enhancer incorporates conversation history to produce a standalone query optimized for vector search. First questions pass through unchanged.

In [ ]:
# ============================================================================
# NODE: Query Enhancement with Conversation Context
# ============================================================================

def enhance_user_query(state: ConversationState):
    """
    Reformulates user questions based on conversation history to create
    standalone queries optimized for vector search.
    """
    print(f" Enhancing query: {state['current_query'].content}")

    # Initialize state for new query processing
    state["retrieved_documents"] = []
    state["topic_relevance"] = ""
    state["enhanced_query"] = ""
    state["should_generate"] = False
    state["optimization_attempts"] = 0

    # Ensure conversation history exists
    if "conversation_history" not in state or state["conversation_history"] is None:
        state["conversation_history"] = []

    # Add current query to history if not already present
    if state["current_query"] not in state["conversation_history"]:
        state["conversation_history"].append(state["current_query"])

    # Check if we have conversation context
    if len(state["conversation_history"]) > 1:
        previous_messages = state["conversation_history"][:-1]
        current_question = state["current_query"].content

        context_messages = [
            SystemMessage(
                content="""You are an expert query reformulator. Transform the user's question into a standalone,
                search-optimized query that incorporates relevant context from the conversation history.

                Guidelines:
                - Make the question self-contained and clear
                - Preserve the user's intent while adding necessary context
                - Optimize for vector database retrieval
                - Keep the reformulated query concise but comprehensive"""
            )
        ]
        context_messages.extend(previous_messages)
        context_messages.append(HumanMessage(content=f"Current question: {current_question}"))

        enhancement_prompt = ChatPromptTemplate.from_messages(context_messages)
        llm = get_llm()

        formatted_prompt = enhancement_prompt.format()
        response = llm.invoke(formatted_prompt)
        enhanced_question = response.content.strip()

        print(f" Enhanced query: {enhanced_question}")
        state["enhanced_query"] = enhanced_question
    else:
        state["enhanced_query"] = state["current_query"].content
        print(f"First query - using original: {state['enhanced_query']}")

    return state

: 

### 4.2 🏷️ Topic Validator - Smart Domain Classification

Uses Pydantic structured output to classify whether the question falls within the TechFlow Solutions domain. Off-topic queries are routed to a polite rejection handler.

In [ ]:
# ============================================================================
# NODE: Topic Validation with Structured Output
# ============================================================================

class TopicRelevance(BaseModel):
    """Structured output for topic classification, validated by Pydantic."""

    classification: Literal["RELEVANT", "IRRELEVANT"] = Field(
        description=(
            "'RELEVANT' if the question is about TechFlow Solutions services, pricing "
            "or company information; 'IRRELEVANT' for anything else."
        )
    )
    confidence: Literal["HIGH", "MEDIUM", "LOW"] = Field(
        description="Confidence in the classification."
    )

def validate_topic_relevance(state: ConversationState):
    """
    Determines if the user's question is within our knowledge domain.
    Uses the enhanced query for better classification accuracy.
    """
    print("Validating topic relevance...")

    classification_prompt = SystemMessage(
        content="""You are a topic classifier for TechFlow Solutions support system.

        RELEVANT topics include:
        - TechFlow Solutions services (cloud infrastructure, migration, DevOps)
        - Pricing for any TechFlow Solutions products or services
        - Company information (history, team, locations)
        - Support procedures and response times
        - Security and compliance features
        - Technical specifications and capabilities

        IRRELEVANT topics include:
        - General technology questions not specific to TechFlow
        - Other companies' products or services
        - Personal questions unrelated to business
        - Weather, news, or general knowledge queries

        Classify based on the enhanced query which incorporates conversation context."""
    )

    user_question = HumanMessage(
        content=f"Enhanced query to classify: {state['enhanced_query']}"
    )

    classification_chain = ChatPromptTemplate.from_messages([classification_prompt, user_question])
    llm = get_llm()

    structured_llm = llm.with_structured_output(TopicRelevance)
    classifier = classification_chain | structured_llm

    result = classifier.invoke({})
    # No .strip()/.upper() needed - Literal guarantees an exact, known value
    state["topic_relevance"] = result.classification

    print(f"Topic classification: {state['topic_relevance']} (Confidence: {result.confidence})")
    return state

### 4.3 📚 Content Retriever - Intelligent Document Fetching

Simple retrieval step that queries the vector store with the enhanced query.

In [ ]:
# ============================================================================
# NODE: Document Retrieval from Vector Store
# ============================================================================

def fetch_relevant_content(state: ConversationState):
    """Retrieves documents from the knowledge base using the enhanced query."""
    print("Fetching relevant documents...")

    retrieved_docs = document_retriever.invoke(state["enhanced_query"])

    print(f"Retrieved {len(retrieved_docs)} documents")
    for i, doc in enumerate(retrieved_docs):
        print(f"   Document {i+1}: {doc.page_content[:50]}...")

    state["retrieved_documents"] = retrieved_docs
    return state

### 4.4 📋 Relevance Assessor - Document Quality Control

Each retrieved document is individually assessed for relevance using structured LLM output. Only documents graded as `RELEVANT` proceed to response generation. If none pass, the query optimizer is triggered.

In [ ]:
# ============================================================================
# NODE: Document Relevance Assessment with Structured Output
# ============================================================================

class DocumentRelevance(BaseModel):
    """Structured output for document relevance assessment"""
    relevance: str = Field(
        description="Is this document relevant to answering the question? Answer 'RELEVANT' or 'IRRELEVANT'"
    )
    reasoning: str = Field(
        description="Brief explanation of why the document is relevant or irrelevant"
    )

def assess_document_relevance(state: ConversationState):
    """
    Evaluates each retrieved document to determine if it's relevant
    for answering the user's question.
    """
    print("Assessing document relevance...")

    assessment_prompt = SystemMessage(
        content="""You are a document relevance assessor. Evaluate whether each document
        contains information that can help answer the user's question.

        A document is RELEVANT if it contains:
        - Direct answers to the question
        - Supporting information that contributes to a complete answer
        - Context that helps understand the topic

        A document is IRRELEVANT if it:
        - Contains no information related to the question
        - Discusses completely different topics
        - Provides no value for answering the question

        Be strict but fair in your assessment."""
    )

    llm = get_llm(model="databricks-gemini-2-5-flash")
    structured_llm = llm.with_structured_output(DocumentRelevance)

    relevant_documents = []

    for i, doc in enumerate(state["retrieved_documents"]):
        assessment_query = HumanMessage(
            content=f"""Question: {state['enhanced_query']}

            Document to assess:
            {doc.page_content}

            Is this document relevant for answering the question?"""
        )

        assessment_chain = ChatPromptTemplate.from_messages([assessment_prompt, assessment_query])
        assessor = assessment_chain | structured_llm

        result = assessor.invoke({})
        print(f"Document {i+1}: {result.relevance} - {result.reasoning}")

        if result.relevance.strip().upper() == "RELEVANT":
            relevant_documents.append(doc)

    state["retrieved_documents"] = relevant_documents
    state["should_generate"] = len(relevant_documents) > 0

    print(f"Final relevant documents: {len(relevant_documents)}")
    return state

### 4.5 💬 Response Generator - Context-Aware Answer Creation

Generates the final response using conversation history and relevant documents. The prompt instructs the LLM to be conversational, cite specific details, and acknowledge limitations.

In [ ]:
# ============================================================================
# NODE: Context-Aware Response Generation
# ============================================================================

def generate_contextual_response(state: ConversationState):
    """Generates final response using conversation history and relevant documents."""
    print("Generating contextual response...")

    if "conversation_history" not in state or state["conversation_history"] is None:
        raise ValueError("Conversation history is required for response generation")

    conversation_context = state["conversation_history"]
    relevant_docs = state["retrieved_documents"]
    enhanced_question = state["enhanced_query"]

    response_template = """You are a knowledgeable TechFlow Solutions support agent. Generate a helpful,
    accurate response based on the conversation history and retrieved documents.

    Guidelines:
    - Use information from the provided documents to answer the question
    - Maintain conversation context and refer to previous exchanges when relevant
    - Be conversational and helpful in tone
    - If the documents don't fully answer the question, acknowledge limitations
    - Provide specific details when available (prices, timeframes, etc.)

    Conversation History:
    {conversation_history}

    Retrieved Knowledge:
    {document_context}

    Current Question: {current_question}

    Generate a helpful response:"""

    response_prompt = ChatPromptTemplate.from_template(response_template)
    llm = get_llm(model="databricks-gemini-2-5-flash")
    response_chain = response_prompt | llm

    response = response_chain.invoke({
        "conversation_history": conversation_context,
        "document_context": relevant_docs,
        "current_question": enhanced_question
    })

    generated_response = response.content.strip()
    state["conversation_history"].append(AIMessage(content=generated_response))

    print(f"Generated response: {generated_response[:100]}...")
    return state

### 4.6 🔄 Query Optimizer - Adaptive Search Improvement

When retrieved documents are deemed irrelevant, this node rewrites the query using synonyms and alternative phrasing, then triggers another retrieval attempt. A maximum of 2 optimization attempts prevents infinite loops.

In [ ]:
# ============================================================================
# NODE: Query Optimization for Failed Retrievals
# ============================================================================

def optimize_search_query(state: ConversationState):
    """
    Refines the search query when initial retrieval doesn't yield relevant results.
    Includes loop prevention to avoid infinite optimization cycles.
    """
    print("Optimizing search query...")

    current_attempts = state.get("optimization_attempts", 0)

    # Prevent infinite optimization loops
    if current_attempts >= 2:
        print("Maximum optimization attempts reached")
        return state

    current_query = state["enhanced_query"]

    optimization_prompt = SystemMessage(
        content="""You are a search query optimizer. The current query didn't retrieve relevant documents.

        Create an improved version that:
        - Uses different keywords or synonyms
        - Adjusts the query structure for better matching
        - Maintains the original intent while improving searchability
        - Considers alternative ways to express the same concept

        Provide only the optimized query without explanations."""
    )

    optimization_request = HumanMessage(
        content=f"Current query that needs optimization: {current_query}"
    )

    optimization_chain = ChatPromptTemplate.from_messages([optimization_prompt, optimization_request])
    llm = get_llm(model="databricks-gemini-2-5-flash")

    formatted_prompt = optimization_chain.format()
    response = llm.invoke(formatted_prompt)
    optimized_query = response.content.strip()

    state["enhanced_query"] = optimized_query
    state["optimization_attempts"] = current_attempts + 1

    print(f"Optimized query (attempt {current_attempts + 1}): {optimized_query}")
    return state

---

## 🔀 Part 5: Workflow Routing & Edge Case Handlers

Conditional routing functions direct the flow based on topic relevance and document quality. Two fallback handlers manage off-topic queries and exhausted optimization attempts gracefully.

In [ ]:
# ============================================================================
# ROUTING: Conditional Edges and Fallback Handlers
# ============================================================================

def route_by_topic(state: ConversationState):
    """Routes based on topic relevance classification"""
    print("Routing based on topic relevance...")
    relevance = state.get("topic_relevance", "").strip().upper()

    if relevance == "RELEVANT":
        print("   -> Proceeding to content retrieval")
        return "fetch_content"
    else:
        print("   -> Routing to off-topic handler")
        return "handle_off_topic"

def route_by_document_quality(state: ConversationState):
    """Routes based on document relevance assessment"""
    print("Routing based on document quality...")
    optimization_attempts = state.get("optimization_attempts", 0)

    if state.get("should_generate", False):
        print("   -> Generating response with relevant documents")
        return "generate_response"
    elif optimization_attempts >= 2:
        print("   -> Maximum optimization attempts reached")
        return "handle_no_results"
    else:
        print("   -> Optimizing query for better results")
        return "optimize_query"

# --- Fallback handlers ---

def handle_off_topic_queries(state: ConversationState):
    """Handles queries outside our knowledge domain"""
    print("Handling off-topic query...")

    if "conversation_history" not in state or state["conversation_history"] is None:
        state["conversation_history"] = []

    off_topic_response = """I'm specialized in helping with TechFlow Solutions services, pricing, and company information.
    Your question seems to be outside my area of expertise.

    I can help you with:
    - Our cloud infrastructure services and pricing
    - Support procedures and response times
    - Company information and team details
    - Security and compliance features

    Is there something specific about TechFlow Solutions I can help you with?"""

    state["conversation_history"].append(AIMessage(content=off_topic_response))
    return state

def handle_no_relevant_results(state: ConversationState):
    """Handles cases where no relevant documents are found after optimization"""
    print("No relevant results found after optimization...")

    if "conversation_history" not in state or state["conversation_history"] is None:
        state["conversation_history"] = []

    no_results_response = """I apologize, but I couldn't find specific information to answer your question in our current knowledge base.

    This might be because:
    - The information isn't available in our documentation
    - Your question might need clarification
    - You might need to contact our support team directly

    For immediate assistance, you can reach our support team at support@techflow.com or call 1-800-TECHFLOW."""

    state["conversation_history"].append(AIMessage(content=no_results_response))
    return state

---

## 🏗️ Part 6: Assemble & Visualize the Graph

We connect all 8 nodes with edges and conditional routing into a complete LangGraph workflow. The graph uses `MemorySaver` for conversation persistence across invocations.

```
enhance_query -> validate_topic -> [on-topic: fetch_content | off-topic: handle_off_topic -> END]
                                          |
                                   assess_relevance -> [relevant: generate_response -> END]
                                                       [not relevant: optimize_query -> fetch_content (retry)]
                                                       [max attempts: handle_no_results -> END]
```

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Assemble the Advanced RAG Workflow
# ============================================================================

# Initialize conversation memory
conversation_memory = MemorySaver()

# Create workflow graph
workflow = StateGraph(ConversationState)

# --- Add all processing nodes ---
workflow.add_node("enhance_query", enhance_user_query)
workflow.add_node("validate_topic", validate_topic_relevance)
workflow.add_node("handle_off_topic", handle_off_topic_queries)
workflow.add_node("fetch_content", fetch_relevant_content)
workflow.add_node("assess_relevance", assess_document_relevance)
workflow.add_node("generate_response", generate_contextual_response)
workflow.add_node("optimize_query", optimize_search_query)
workflow.add_node("handle_no_results", handle_no_relevant_results)

# --- Define workflow connections ---
workflow.add_edge("enhance_query", "validate_topic")

# Route by topic relevance
workflow.add_conditional_edges(
    "validate_topic",
    route_by_topic,
    {"fetch_content": "fetch_content", "handle_off_topic": "handle_off_topic"},
)

# Content processing pipeline
workflow.add_edge("fetch_content", "assess_relevance")

# Route by document quality
workflow.add_conditional_edges(
    "assess_relevance",
    route_by_document_quality,
    {
        "generate_response": "generate_response",
        "optimize_query": "optimize_query",
        "handle_no_results": "handle_no_results",
    },
)

# Optimization loop
workflow.add_edge("optimize_query", "fetch_content")

# Terminal edges
workflow.add_edge("generate_response", END)
workflow.add_edge("handle_no_results", END)
workflow.add_edge("handle_off_topic", END)

# Set entry point and compile
workflow.set_entry_point("enhance_query")
advanced_rag_agent = workflow.compile(checkpointer=conversation_memory)

print("✅ Advanced RAG agent compiled with 8 nodes!")

In [ ]:
# ============================================================================
# GRAPH VISUALIZATION: Display the Agent Workflow
# ============================================================================

display(
    Image(
        advanced_rag_agent.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

---

## 🚀 Part 7: Test the Agent

We run 5 test scenarios to validate different aspects of the agent:
1. **Off-topic rejection** - Weather question should be filtered
2. **On-topic retrieval** - Pricing query should return structured answer
3. **Follow-up with context** - Same session, different topic
4. **Company information** - Direct factual query
5. **Context-dependent follow-up** - Requires conversation history to understand

In [ ]:
# ============================================================================
# TEST 1: Off-Topic Query (should be filtered)
# ============================================================================

print("=== Test 1: Off-Topic Query ===")
test_input = {"current_query": HumanMessage(content="What's the weather like today?")}
result = advanced_rag_agent.invoke(
    input=test_input,
    config={"configurable": {"thread_id": "test_session_1"}}
)
print(f"Response: {result['conversation_history'][-1].content}")

In [ ]:
# ============================================================================
# TEST 2: On-Topic Pricing Query
# ============================================================================

print("=== Test 2: Service Pricing Query ===")
test_input = {"current_query": HumanMessage(content="What are your support service pricing options?")}
result = advanced_rag_agent.invoke(
    input=test_input,
    config={"configurable": {"thread_id": "test_session_2"}}
)
print(f"Response: {result['conversation_history'][-1].content}")

In [ ]:
# ============================================================================
# TEST 3: Follow-Up Question (same session as Test 2)
# ============================================================================

print("=== Test 3: Follow-Up Question ===")
test_input = {"current_query": HumanMessage(content="What about the infrastructure services?")}
result = advanced_rag_agent.invoke(
    input=test_input,
    config={"configurable": {"thread_id": "test_session_2"}}  # Same session
)
print(f"Response: {result['conversation_history'][-1].content}")

In [ ]:
# ============================================================================
# TEST 4: Company Information Query
# ============================================================================

print("=== Test 4: Company Information ===")
test_input = {"current_query": HumanMessage(content="Who founded TechFlow Solutions?")}
result = advanced_rag_agent.invoke(
    input=test_input,
    config={"configurable": {"thread_id": "test_session_3"}}
)
print(f"Response: {result['conversation_history'][-1].content}")

In [ ]:
# ============================================================================
# TEST 5: Context-Dependent Follow-Up (same session as Test 4)
# ============================================================================

print("=== Test 5: Context-Dependent Follow-Up ===")
test_input = {"current_query": HumanMessage(content="How does their support compare to the enterprise level?")}
result = advanced_rag_agent.invoke(
    input=test_input,
    config={"configurable": {"thread_id": "test_session_3"}}  # Same session as company info
)
print(f"Response: {result['conversation_history'][-1].content}")

---

## 📝 Summary

In this notebook, we built an **Advanced RAG Agent** with 8 graph nodes and sophisticated routing:

### 1. Query Enhancement
- Conversation history is folded into standalone queries for better vector search
- Follow-up questions are automatically contextualized

### 2. Topic Validation & Routing
- **Structured output** (Pydantic) classifies queries as RELEVANT or IRRELEVANT
- Off-topic queries are handled gracefully without wasting retrieval resources

### 3. Document Relevance Assessment
- Each retrieved document is individually graded before use
- Only relevant documents proceed to response generation

### 4. Query Optimization Loop
- Failed retrievals trigger automatic query rewriting (max 2 attempts)
- Prevents infinite loops while maximizing retrieval success

### 5. Conversation Memory
- `MemorySaver` checkpointer enables multi-turn conversations
- Thread IDs isolate independent conversation sessions

### Next Steps
- **05_RAG_as_Tool_in_Agents** - RAG wrapped as a tool the LLM decides when to invoke
- **04_Agents/** - Apply these patterns to real-world research and intelligence agents